# ECON 524 — Big Data Econometrics
## Computational Assignment 2: Nonlinear Machine Learning (Trees, Bagging, Random Forests, Neural Networks)

**Emory University · Fall 2026**  
Instructor: Zheng Fang · Lab instructor: Joel Reyes Mora · Grader: Marcelo Ortiz-Villavicencio

**Time allowed:** two weeks from release. **Based on:** Lecture 2 (Nonlinear Machine Learning), §1.1–1.2 and §2.1–2.2.

**Name:** ______________________   **Collaborators (if any):** ______________________

## Instructions

1. **Submit this notebook** with every cell executed from top to bottom (*Kernel → Restart & Run All*) and all output visible.
2. **Written work goes in Markdown cells.** This is a computational assignment: no proofs are required, but every answer must be supported by output produced in your notebook.
3. **Every figure or table must be followed by an interpretation** that answers the question asked, not a description of the plot. Interpretation carries roughly half the credit of each part.
4. **Reproducibility.** Fix and report random seeds. Results must be reproducible on a laptop CPU; no part of this assignment requires a GPU.
5. **Allowed software.** NumPy, pandas, SciPy, matplotlib, scikit-learn, and PyTorch. Where a question says *implement yourself*, you may use a library **only** for the building block named in that question.
6. **AI use.** Follow the course README's responsible-use framework and include the required acknowledgment statement at the end of the notebook.

**Ground rules on accuracy.** All data are simulated. Because you know the true regression function $g$, report accuracy as *excess risk*
$$\text{MSE}(\hat f) \;=\; \frac{1}{n_{test}}\sum_{i=1}^{n_{test}}\big(\hat f(Z_i)-g(Z_i)\big)^2$$
on a large independent test sample ($n_{test}=20{,}000$) unless a question says otherwise. For classification, report the test misclassification rate.

## Simulation designs

| Name | Design | Training size |
|---|---|---|
| **DGP-1** (univariate, smooth) | $Z\sim U[0,1]$, $\;Y=e^{4Z}+\varepsilon$, $\;\varepsilon\sim N(0,2^2)$. The *noiseless* version sets $\varepsilon=0$. | $n=500$ |
| **DGP-2** (checkerboard) | $Z\sim U[0,1]^2$, $\;Y=4\cdot\mathbf 1\{(Z_1>0.5)\neq(Z_2>0.5)\}+\varepsilon$, $\;\varepsilon\sim N(0,1)$ | $n=2{,}000$ |
| **DGP-3** (correlated predictors) | $p=20$; $Z_j=\sqrt{0.7}\,F+\sqrt{0.3}\,U_j$ with $F,U_1,\dots,U_{20}\overset{iid}{\sim}N(0,1)$; $\;g(z)=3\tanh(z_1)+0.6\sum_{j=2}^{11}\sin(z_j)$; $\;Y=g(Z)+\varepsilon$, $\;\varepsilon\sim N(0,1)$. Binary version: $D=\mathbf 1\{Y>0\}$. | $n=1{,}000$ |
| **DGP-4** (sparse, with interaction; Friedman, 1991) | $Z\sim U[0,1]^p$, $p\in\{10,50\}$; $\;g(z)=10\sin(\pi z_1z_2)+20(z_3-0.5)^2+10z_4+5z_5$; $\;\varepsilon\sim N(0,1)$ | $n=1{,}000$ |

Note that in DGP-3 predictors $12$–$20$ are irrelevant, and in DGP-4 only the first $5$ predictors matter.

## Setup

Use the cell below for imports, seeds, and your data-generating functions.

# Question 1 — CART from first principles

### Q1(a) The splitting criterion

- **Computation (split search).** Using DGP-1, *implement yourself* an exhaustive search over all candidate split points for a single split, using the criterion on the slides: choose the split that maximizes $\frac{1}{n_L}\big(\sum_{i\in I_L}Y_i\big)^2+\frac{1}{n_R}\big(\sum_{i\in I_R}Y_i\big)^2$. Verify that your split matches the root split of a library regression tree of depth 1, and plot the criterion against the candidate split point.
- **Computation (what the split trades off).** For every candidate split, compute the reduction in the in-sample sum of squared errors (parent SSE minus the two children's SSE) and check numerically that it equals $\frac{n_Ln_R}{n}(\bar Y_L-\bar Y_R)^2$. Plot the two ingredients — the *balance* term $n_Ln_R/n$ and the *separation* term $(\bar Y_L-\bar Y_R)^2$ — against the split point. Separately, plot the in-sample MSE and the test excess risk of trees of depth $1,\dots,12$.
- **Interpret.** Your split will not fall at the median of $Z$. Using your two-ingredient plot, explain which two forces the split balances and why the shape of $e^{4z}$ pushes the split where it lands. What does this tell you about where a tree "spends" its splits? What does your depth plot say about using in-sample fit to choose tree size?

**Your interpretation:**

*Write your interpretation here.*

### Q1(b) Why grow-then-prune?

- **Computation (stopping).** On DGP-2, grow regression trees that stop splitting a node when the best split reduces the in-sample MSE by less than $1\%$ of $\widehat{\text{Var}}(Y)$; repeat with $5\%$. Report the number of leaves and the test excess risk.
- **Computation (pruning).** Grow a large tree (minimum 5 observations per leaf), then prune it by cost-complexity pruning, eq. (1) in the slides, choosing $\alpha$ by 15-fold cross-validation as in Algorithm 0. Report the chosen $\alpha$, the number of leaves, and the test excess risk of (i) the stopped trees, (ii) the unpruned tree, (iii) the pruned tree. Plot the CV-MSE against $\alpha$ and $|T|$ against $\alpha$.
- **Interpret.** Explain the slide's claim that threshold stopping is "short-sighted" using the geometry of DGP-2: what is the expected MSE reduction from the best *single* split at the root? Why does pruning succeed where stopping fails? Relate the shape of your CV curve to the bias–variance trade-off.

**Your interpretation:**

*Write your interpretation here.*

### Q1(c) Impurity measures for classification

- **Computation (impurity functions).** Write functions for the three node-impurity measures on the slides (misclassification error, Gini index, entropy) and plot them against $\hat p_{m1}$ for a two-class node. Then consider a node with 400 observations of each class and two candidate splits: split A produces children $(300,100)$ and $(100,300)$; split B produces $(200,400)$ and $(200,0)$. Compute the weighted impurity of each split under each measure and report which split each measure prefers.
- **Computation (classification trees).** On the binary version of DGP-3, grow classification trees using (i) the Gini index and (ii) entropy, and choose the pruning penalty by 15-fold CV using misclassification error as the CV criterion. Report the number of leaves and the test misclassification rate of each, and compare with the Bayes error rate, which you can compute because you know the DGP ($\Pr(D=1\mid Z)=\Phi(g(Z))$).
- **Interpret.** Using the shapes in your impurity plot, explain why misclassification error cannot tell splits A and B apart while Gini and entropy can, and why the lecture recommends Gini/entropy for *growing* a tree but misclassification error for *pruning* it. Does the choice between Gini and entropy matter much in your simulation? How far is the pruned tree from the Bayes error, and what does that gap tell you?

**Your interpretation:**

*Write your interpretation here.*

# Question 2 — Instability of a single tree and the logic of bagging

### Q2(a) One tree is a noisy estimator

- **Computation (approximation).** On the noiseless DGP-1 with $n=100$, plot the fits of a depth-2 tree, an unpruned tree, and a random forest together with the true $g$ (a replication of Figures 9.6–9.8 in Chernozhukov et al., 2024).
- **Computation (instability).** Draw $R=200$ independent training samples from DGP-1. For each, fit an unpruned tree and record (i) its predictions at $z_0\in\{0.3,0.8\}$ and (ii) its root split point. Show histograms and report the Monte Carlo variance of the predictions at each $z_0$.
- **Interpret.** Connect your results to the slide's two criticisms of CART (high variance; poor approximation of the CEF). Which criticism is about bias and which about variance? Compare the variability of the root split with the variability of the predictions at $z_0$: which is more stable, and what does this tell you about *where* in the tree the instability comes from?

**Your interpretation:**

*Write your interpretation here.*

### Q2(b) Bagging, implemented yourself

- **Computation.** *Implement yourself* bootstrap aggregation of unpruned regression trees (a library may be used only to fit each individual tree).
- **Computation.** Using the same $R=200$ Monte Carlo samples as in Q2(a), compute the squared bias, variance, and MSE of the bagged prediction at each $z_0$ for $B\in\{1,5,25,100\}$. Present the results in a table.
- **Interpret.** Which component does bagging reduce and which does it leave essentially unchanged, and why? Why is the $B=1$ bagged tree not the same estimator as the single tree of Q2(a)? Using your noiseless plot from Q2(a), explain why averaging step functions with *different* cut-points produces a smoother fit.

**Your interpretation:**

*Write your interpretation here.*

### Q2(c) Why the variance stops falling

- **Computation.** From your Monte Carlo output in Q2(b), estimate at each $z_0$ (i) $\sigma^2$, the variance of a *single* bagged tree's prediction across training samples, and (ii) $\rho$, the average correlation between the predictions of two different trees in the same bagged ensemble.
- **Computation.** The lecture's statement that correlated trees limit the benefit of averaging can be quantified by the formula $\text{Var}\big(\frac1B\sum_b\hat f^b\big)=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2$. Plug in your estimates, overlay the formula's curve (for $B=1,\dots,100$) on the Monte Carlo variance of the bagged prediction, and report the value the variance approaches as $B$ grows.
- **Interpret.** Where does the correlation $\rho$ between two bagged trees come from, given that each uses a different bootstrap sample? What limits the variance reduction from adding trees, and why does this motivate the random forest?

**Your interpretation:**

*Write your interpretation here.*

# Question 3 — Random forests: decorrelation, tuning, and dimensionality

### Q3(a) The role of $q$

- **Computation.** On DGP-3, fit random forests with $B=200$ trees and $q\in\{1,2,4,7,10,20\}$ predictors considered at each split ($q=20$ is bagging). Report the test excess risk averaged over 5 independent training samples.
- **Computation.** Using $R\ge 50$ Monte Carlo training samples, estimate at 100 fixed test points the single-tree variance $\sigma^2(q)$, the between-tree correlation $\rho(q)$, their product $\rho(q)\sigma^2(q)$, and the squared bias. Present these in a table.
- **Interpret.** Describe the shape of excess risk as a function of $q$. Assess the slide's statement "smaller $q$: more randomness, less variance/overfitting but more bias" against your estimates. Does decorrelation come for free? Why does the equicorrelated structure of DGP-3 favor random forests over bagging?

**Your interpretation:**

*Write your interpretation here.*

### Q3(b) Number of trees and software defaults

- **Computation.** Plot the test excess risk against the number of trees $B$ from 1 to 1,000, for bagging and for the best $q$ from Q3(a).
- **Computation.** Report the default values of $q$ and $B$ in the library you use, for both regression and classification forests.
- **Interpret.** Use the formula from Q2(c) to explain the shape of both curves. Can adding trees *hurt* regression MSE in expectation? In light of the lecture's distinction between bagging and random forests, state precisely which method the library's regression default implements.

**Your interpretation:**

*Write your interpretation here.*

### Q3(c) Many irrelevant predictors

- **Computation (wasted splits).** In DGP-4 only the first 5 predictors matter. For $p\in\{10,50\}$ and each $q$ in your grid, *simulate* 100,000 random draws of $q$ predictors out of $p$ and record the share of draws that contain none of the 5 relevant predictors. Then, in forests fitted to DGP-4, compute the share of splits made on irrelevant predictors, both among all splits (internal nodes, across all trees) and among the splits in the top three levels of each tree.
- **Computation (performance).** Fit random forests on DGP-4 with $p=10$ and $p=50$ over a grid of $q$ values (include $q=p$), averaging the test excess risk over 3 training samples. Then keep $n=1{,}000$ fixed and let $p\in\{5,10,25,50,100\}$ (so $0$ to $95$ irrelevant predictors are added); for each $p$ compare bagging ($q=p$) with a random forest using $q=\max\{1,\text{round}(p/3)\}$ and plot test excess risk against $p$.
- **Interpret.** Using your simulated shares and split-usage shares, explain why small $q$ performs poorly when $p=50$. Why does the share among *all* splits barely change with $q$, while the share in the top levels does? Are irrelevant predictors "free" for a random forest or for bagging? What does this suggest for empirical work in economics where a researcher is tempted to throw every available control variable into a machine-learning model?

**Your interpretation:**

*Write your interpretation here.*

# Question 4 — Training neural networks

### Q4(a) What a shallow network can represent

- **Computation (ReLU networks).** Fit ReLU SNNs with $k\in\{1,2,4,8,16,64\}$ hidden units to the noiseless DGP-1 ($n=500$) and plot the fits. Each ReLU unit $\max\{\omega_{j0}+\omega_{j1}z,0\}$ changes slope at $z=-\omega_{j0}/\omega_{j1}$: extract these "kink" locations from your fitted weights, count how many fall inside $[0,1]$, and mark them on the plot for $k=8$.
- **Computation (comparisons).** For each $k$, fit a regression tree with $k+1$ leaves (so both have $k$ break points) and tabulate the approximation errors of the two methods. Then replace ReLU by the activation $\sigma(u)=u^2$, fit SNNs with $k\in\{2,8,64\}$, and compare their errors with that of the best quadratic polynomial in $z$ (OLS of $Y$ on $1,z,z^2$).
- **Interpret.** How does approximation error change with $k$, and why does a function built from line segments approximate $e^{4z}$ better than one built from flat steps with the same number of break points? Did your fitted networks use all of their kinks? What does the $u^2$ experiment show, and how does it relate to condition (ii) of the universal approximation theorem on the slides?

**Your interpretation:**

*Write your interpretation here.*

### Q4(b) Gradient descent, implemented yourself

- **Computation (your own optimizer).** For a ReLU SNN with $k=8$ on DGP-1 and loss $R(\theta)=\frac1n\sum_i\{Y_i-f(Z_i;\theta)\}^2$, *implement yourself* the gradient-descent update $\theta^{(t)}\leftarrow\theta^{(t-1)}-\rho\,\nabla R(\theta^{(t-1)})$: compute the gradient by automatic differentiation (backpropagation), but write the update step yourself without using a built-in optimizer. Check the automatic-differentiation gradient against finite differences. Implement both full-batch gradient descent and mini-batch SGD (batch size 32, reshuffling every epoch).
- **Computation (learning rates and initialization).** From one random initialization (all weights and biases drawn from $N(0,1)$, output weights from $N(0,1/k)$), train with learning rates $\rho\in\{10^{-4},10^{-3},10^{-2},5\times10^{-2}\}$ and report the loss path and the excess risk. For each run, count the hidden units that are inactive (pre-activation $\le0$) at *every* training observation, at initialization and at the end. Finally, set **every** parameter to zero, as in the algorithm on the slides: print the gradient of each parameter at the first step, train, and report the fitted function.
- **Interpret.** Explain what happens at each learning rate, using the idea on the slides that gradient descent takes a step within a small neighborhood of the current $\theta$. Using the gradients you printed, explain which parameters move under the zero initialization and what function the network ends up fitting. What do the inactive units tell you? Why do the slides insist that it is "important to try different values" of $\theta^{(0)}$?

**Your interpretation:**

*Write your interpretation here.*

### Q4(c) The full training pipeline

- **Computation (training and ablations).** Using one DGP-3 training sample of $n=1{,}000$ split into 800 training and 200 validation observations, train the two-hidden-layer MLP from the slides (64 and 32 ReLU units, dropout $0.2$ after the first hidden layer) with AdamW (learning rate $10^{-3}$, weight decay $10^{-4}$), mini-batches of 64 reshuffled every epoch, at most 300 epochs, and early stopping with patience 10 (Algorithm 1). Plot training and validation loss by epoch. Then run three ablations, each over 3 seeds: (i) no early stopping (keep the epoch-300 weights), (ii) no dropout, (iii) weight decay set to zero.
- **Computation (weight decay and comparison).** For $\lambda\in\{10^{-4},10^{-2},10^{-1}\}$, train (a) AdamW with weight decay $\lambda$ and (b) Adam with the penalty $\frac{\lambda}{2}\|\theta\|_2^2$ added to the loss (3 seeds each), and report the final squared norm of the weights and the test excess risk. Then compare the test excess risk of your best network with your best random forest from Q3(a), with bagging, and with OLS (the best *linear* predictor), all trained on the same $n=1{,}000$ observations.
- **Interpret.** Where does the validation curve turn, and what is early stopping regularizing? Which ablation hurts most? How do AdamW and "Adam plus penalty" differ in how much they actually shrink the weights, and why does the slides' default weight decay barely matter? Which learner wins, and would you expect the ranking to hold in other designs?

**Your interpretation:**

*Write your interpretation here.*

## References

- Breiman, L. (1996). Bagging predictors. *Machine Learning*, 24, 123–140.
- Breiman, L. (2001). Random forests. *Machine Learning*, 45, 5–32.
- Breiman, L., Friedman, J., Olshen, R., and Stone, C. (1984). *Classification and Regression Trees*. Chapman & Hall/CRC.
- Chernozhukov, V., Hansen, C., Kallus, N., Spindler, M., and Syrgkanis, V. (2024). *Applied Causal Inference Powered by ML and AI*. Online manuscript, Ch. 9.
- Cybenko, G. (1989). Approximation by superpositions of a sigmoidal function. *Math. Control Signals Systems*, 2, 303–314.
- Friedman, J. H. (1991). Multivariate adaptive regression splines. *Annals of Statistics*, 19, 1–67.
- Hastie, T., Tibshirani, R., and Friedman, J. (2009). *The Elements of Statistical Learning*, 2nd ed., §9.2 and §15.2.
- Ho, T. K. (1995). Random decision forests. *Proc. 3rd ICDAR*, 278–282.
- Hornik, K., Stinchcombe, M., and White, H. (1989). Multilayer feedforward networks are universal approximators. *Neural Networks*, 2, 359–366.
- James, G., Witten, D., Hastie, T., Tibshirani, R., and Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*, Ch. 8 and 10.
- Kingma, D. P. and Ba, J. (2015). Adam: A method for stochastic optimization. *ICLR*.
- Leshno, M., Lin, V. Y., Pinkus, A., and Schocken, S. (1993). Multilayer feedforward networks with a nonpolynomial activation function can approximate any function. *Neural Networks*, 6, 861–867.
- Loshchilov, I. and Hutter, F. (2019). Decoupled weight decay regularization. *ICLR*.
- Oshiro, T. M., Perez, P. S., and Baranauskas, J. A. (2012). How many trees in a random forest? *MLDM*, 154–168.
- Probst, P. and Boulesteix, A.-L. (2018). To tune or not to tune the number of trees in random forest. *JMLR*, 18, 1–18.


## AI-use acknowledgment

*Insert the acknowledgment statement required by the course README here.*